# HomeWork 2 
## Luke O'Neill

# 1: Decoder-based Transformer

In [1]:
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

Looking in indexes: https://download.pytorch.org/whl/cpu
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn

In [3]:
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_hidden_dim, dropout):
        super(DecoderBlock, self).__init__()

        self.attention = nn.MultiheadAttention(d_model, num_heads , dropout = dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1= nn.Dropout(dropout)
        self.linear1 = nn.Linear(d_model, ff_hidden_dim)
        self.linear2 = nn.Linear(ff_hidden_dim, d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)


    def forward(self, x, tgt_mask):
        attn_output, _ = self.attention(x,x,x,attn_mask = tgt_mask)
        x = x + self.dropout1(attn_output)
        x = self.norm1(x)
        ff_output = self.linear2(F.relu(self.linear1(x)))
        x = x + self.dropout2(ff_output)
        x = self.norm2(x)
        return x 

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout = .1, max_len = 5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype = torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0,1)
        self.register_buffer("pe", pe)


    def forward(self, x):
        x= x + self.pe[:x.size(0),:]
        return self.dropout(x)

In [4]:
def generate_square_subsequent_mask(size):
    mask = (torch.triu(torch.ones(size, size)) == 1).transpose(0,1)
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask

In [5]:
class TransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ff_hidden_dim, dropout):
        super(TransformerDecoder, self).__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        self.transformer_block = DecoderBlock(d_model, num_heads, ff_hidden_dim, dropout)
        self.linear = nn.Linear(d_model, vocab_size)
        self.softmax = nn.LogSoftmax(dim = -1)

    def forward(self, x) :
        x= self.embedding(x)
        x = self.pos_encoder(x)
        tgt_mask = generate_square_subsequent_mask(x.size(0))
        x= self.transformer_block(x, tgt_mask)
        output = self.linear(x)
        output = self.softmax(output)
        return output

In [6]:
# hyper parameters

vocab_size = 1000
d_model = 512
num_heads = 1
ff_hidden_dim = 2* d_model
dropout = .01
num_layers = 10
context_length = 50
batch_size = 1


model = TransformerDecoder(vocab_size, d_model, num_heads, ff_hidden_dim, dropout)

input_tensor = torch.randint(0, vocab_size,(context_length, batch_size))

output = model(input_tensor)

predicted_indicies = output.argmax(dim=-1)





# 2: Total Number of HyperParameters

In [7]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print (count_parameters(model))

3127784


# 3: Multiple Decoder based Transformer

In [8]:
class MultiLayerTransformerDecoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, ff_hidden_dim, dropout, num_layers):
        super(MultiLayerTransformerDecoder, self).__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        self.transformer_blocks = nn.ModuleList([
            DecoderBlock(d_model, num_heads, ff_hidden_dim, dropout)
            for _ in range(num_layers)
        ])
        self.linear = nn.Linear(d_model, vocab_size)
        self.softmax = nn.LogSoftmax(dim=-1)


    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoder(x)
        for transformer_block in self.transformer_blocks:
            tgt_mask = generate_square_subsequent_mask(x.size(0))
            x = transformer_block(x, tgt_mask)
        output = self.linear(x)
        output = self.softmax(output)
        return output

In [9]:
import os
os.environ["WANDB_DISABLED"] = "true"

!pip install fsspec==2023.6.0

In [10]:
# hyperParameters
vocab_size = 10000
# reducing this to keep my kernal from crashing
d_model = 512
num_heads = 4
ff_hidden_dim = 4*d_model
dropout = .1
num_layers = 10
context_size = 100
batch_size = 10

input_tensor = torch.randint(0, vocab_size, (context_length, batch_size))
model = MultiLayerTransformerDecoder(vocab_size, d_model, num_heads, ff_hidden_dim, dropout, num_layers)



# 4: Number of HyperParameters in Multi Layer Transformer Decoder Model

In [11]:
print(count_parameters(model))

41773840


# 5: Fool Me Once Paragraph

In [12]:
input_sentence = "fool me once shame on shame on you cannot fool me again"


vocab = ['there', 'is', 'an', 'old', 'saying', 'in', 'tennessee', 'I', 'know', 'it', 'is', 'in', 'texas', 'probably', 'in', 'tennessee', 'that', 'says', 'fool', 'me', 'once', 'shame', 'on', 'shame', 'on', 'you', 'fool', 'me', 'you', "can't", 'get', 'fooled', 'again']
vocab = vocab[-20:]


# hyperparameters 
d_model = 100
num_heads = 1
ff_hidden_dim = 4 * d_model
dropout = .1
num_layers = 4
context_length = 5
batch_size = 1
vocab_size = len(vocab)

word2id = {word: id for id, word in enumerate(vocab)}

id2word = {id:word for id, word in enumerate(vocab)}

model = MultiLayerTransformerDecoder(vocab_size, d_model, num_heads, ff_hidden_dim, dropout, num_layers)


sequence = input_sentence.split()[:context_length]

input_tensor = torch.tensor([[word2id[word] for word in sequence]])

generated_words = []

for i in range(57):
    output = model(input_tensor)
    predicted_index = output.argmax(dim =-1)[0,1]
    predicted_word = id2word[predicted_index.item()]
    print(predicted_word, end = " ")
    generated_words.append(predicted_word)
    input_tensor = torch.cat([input_tensor, predicted_index.unsqueeze(0).unsqueeze(0)],dim = -1)
    time.sleep(.5)

that shame shame that that that that fool that shame that that that shame that shame fool that shame that that that that fool that that that shame that that that that that that that fool that that that that fool fool shame fool that fool tennessee that shame fool shame shame that fool that tennessee shame 

# 6: PreTrained Decoder

In [13]:
%pip install transformers sentencepiece accelerate

cache_dir = "./llm_architecture/deployment"



Note: you may need to restart the kernel to use updated packages.


In [14]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
inputs = tokenizer("Hello world", return_tensors = "pt")
with torch.no_grad():
    out = model(**inputs)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [15]:
# tokenizer_large = GPT2Tokenizer.from_pretrained("gpt2-XL", cache_dir = cache_dir+"/models")
# model_large = GPT2LMHeadModel.from_pretrained("gpt2-XL", cache_dir = cache_dir+"/models")

In [28]:
def generate_text(prompt, cache_dir = cache_dir, model_size = "small", max_tokens = 250, delay = .1):
    model_name = "gpt2"
    tokenizer = GPT2Tokenizer.from_pretrained(model_name, cache_dir =f"{cache_dir}/models")
    model  = GPT2LMHeadModel.from_pretrained(model_name, cache_dir = f"{cache_dir}/models")

    inputs = tokenizer.encode(prompt,return_tensors = 'pt')
    attention_mask = torch.ones(inputs.shape, dtype = torch.long)
    pad_token_id = tokenizer.eos_token_id

    print(prompt + "\n", end = " " , flush = True)
    for _ in range(max_tokens):
        outputs = model.generate(
            inputs, 
            max_length = inputs.shape[-1] + 1,
            do_sample = True,
            pad_token_id = pad_token_id,
            attention_mask=attention_mask
        )
        generated_token = outputs[0][-1].unsqueeze(0).unsqueeze(0)
        generated_word = tokenizer.decode(generated_token[0])
        print(generated_word, end = " " , flush = True)
        inputs = torch.cat([inputs, generated_token],dim = -1)
        attention_mask = torch.cat([attention_mask, torch.ones((1,1), dtype = torch.long)], dim = -1)
        time.sleep(delay)

In [29]:
prompt = "it is always sunny in Philidelphia"

generate_text(prompt, max_tokens = 50)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

it is always sunny in Philidelphia
 ."  For  some  a  time  the  city  sought  a  way  to  replace  one  of  its  former  buildings .  They  needed  a  place  to  park  and  use  the  beach  instead  of  a  park ,  so  they  built  a  concrete  deck  in  the  old  parking  lot  around  10  p . m . 